# LIVE — YOLOv8s (exp6) + MobileNetV3 Gaze 학습 실시간 모니터링

이 노트북을 실행하면 학습 곡선이 **자동으로 갱신**됩니다.  
아래 셀 두 개를 각각 실행하세요. 중단하려면 **■ (Stop)** 버튼을 누르세요.

| 셀 | 모니터링 대상 |
|---|---|
| 셀 1 | YOLOv8s exp6 — 100k 균형 학습 |
| 셀 2 | MobileNetV3 Gaze Estimator — 418k 학습 |

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import clear_output
from pathlib import Path

# ── 경로 설정 ──────────────────────────────────────────────────
EXP_DIR     = Path('C:/Users/SSAFY/Desktop/GAZE-CAPTURE/GazeCapture/MPIIGAZE/runs/yolo/exp6')
RESULTS_CSV = EXP_DIR / 'results.csv'
TOTAL_EPOCHS = 50
REFRESH_SEC  = 30

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
    'font.size':        11,
})

COLORS = {
    'train': '#58a6ff',
    'val':   '#f78166',
    'map50': '#3fb950',
    'map':   '#d2a8ff',
    'lr':    '#ffa657',
    'prec':  '#79c0ff',
    'rec':   '#ffb3ba',
}


def plot_yolo_dashboard(df: pd.DataFrame, total_epochs: int = 50):
    ep     = df['epoch'].values
    n      = len(df)
    latest = df.iloc[-1]

    fig = plt.figure(figsize=(18, 11))
    fig.patch.set_facecolor('#0d1117')
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

    pct     = n / total_epochs * 100
    eta_sec = latest['time'] / n * (total_epochs - n) if n > 0 else 0
    eta_min = eta_sec / 60
    fig.suptitle(
        f'YOLOv8s Eye Detection (exp6)  |  Epoch {n}/{total_epochs}  ({pct:.0f}%)  |  '
        f'ETA {eta_min:.0f} min  |  mAP50 {latest["metrics/mAP50(B)"]:.4f}',
        fontsize=14, fontweight='bold', color='#f0f6fc', y=0.98
    )

    ax0 = fig.add_subplot(gs[0, :])
    ax0.barh(0, 100, color='#21262d', height=0.6)
    ax0.barh(0, pct, color='#238636', height=0.6)
    ax0.set_xlim(0, 100); ax0.set_yticks([])
    ax0.set_xlabel('에포크 진행률 (%)')
    for spine in ax0.spines.values(): spine.set_visible(False)
    ax0.text(50, -0.75,
        f"mAP50: {latest['metrics/mAP50(B)']:.4f}   "
        f"mAP50-95: {latest['metrics/mAP50-95(B)']:.4f}   "
        f"Precision: {latest['metrics/precision(B)']:.4f}   "
        f"Recall: {latest['metrics/recall(B)']:.4f}   "
        f"LR: {latest['lr/pg0']:.5f}",
        ha='center', va='top', color='#c9d1d9', fontsize=10)

    def fill_plot(ax, x, y_tr, y_va, title, ylabel):
        ax.plot(x, y_tr, color=COLORS['train'], lw=2, label='Train')
        ax.plot(x, y_va, color=COLORS['val'],   lw=2, label='Val', linestyle='--')
        ax.fill_between(x, y_tr, alpha=0.15, color=COLORS['train'])
        ax.fill_between(x, y_va, alpha=0.15, color=COLORS['val'])
        ax.set_title(title, fontsize=11, pad=6)
        ax.set_ylabel(ylabel, fontsize=9); ax.set_xlabel('Epoch', fontsize=9)
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
        if len(x) > 1: ax.set_xlim(x[0], max(x[-1], total_epochs))

    ax1 = fig.add_subplot(gs[1, 0])
    fill_plot(ax1, ep, df['train/box_loss'], df['val/box_loss'], 'Box Loss', 'Loss')
    ax2 = fig.add_subplot(gs[1, 1])
    fill_plot(ax2, ep, df['train/cls_loss'], df['val/cls_loss'], 'Cls Loss', 'Loss')
    ax3 = fig.add_subplot(gs[1, 2])
    fill_plot(ax3, ep, df['train/dfl_loss'], df['val/dfl_loss'], 'DFL Loss', 'Loss')

    ax4 = fig.add_subplot(gs[2, 0])
    ax4.plot(ep, df['metrics/mAP50(B)'],    color=COLORS['map50'], lw=2.5, label='mAP50')
    ax4.plot(ep, df['metrics/mAP50-95(B)'], color=COLORS['map'],   lw=2.5, label='mAP50-95', linestyle='--')
    ax4.fill_between(ep, df['metrics/mAP50(B)'],    alpha=0.2, color=COLORS['map50'])
    ax4.fill_between(ep, df['metrics/mAP50-95(B)'], alpha=0.2, color=COLORS['map'])
    ax4.set_title('mAP (눈 탐지 정확도)', fontsize=11, pad=6)
    ax4.set_ylabel('mAP', fontsize=9); ax4.set_xlabel('Epoch', fontsize=9)
    ax4.set_ylim(0, 1); ax4.legend(fontsize=8); ax4.grid(True, alpha=0.3)
    if len(ep) > 1: ax4.set_xlim(ep[0], max(ep[-1], total_epochs))

    ax5 = fig.add_subplot(gs[2, 1])
    ax5.plot(ep, df['metrics/precision(B)'], color=COLORS['prec'], lw=2.5, label='Precision')
    ax5.plot(ep, df['metrics/recall(B)'],    color=COLORS['rec'],  lw=2.5, label='Recall', linestyle='--')
    ax5.fill_between(ep, df['metrics/precision(B)'], alpha=0.15, color=COLORS['prec'])
    ax5.fill_between(ep, df['metrics/recall(B)'],    alpha=0.15, color=COLORS['rec'])
    ax5.set_title('Precision / Recall', fontsize=11, pad=6)
    ax5.set_ylabel('Score', fontsize=9); ax5.set_xlabel('Epoch', fontsize=9)
    ax5.set_ylim(0, 1); ax5.legend(fontsize=8); ax5.grid(True, alpha=0.3)
    if len(ep) > 1: ax5.set_xlim(ep[0], max(ep[-1], total_epochs))

    ax6 = fig.add_subplot(gs[2, 2])
    ax6.plot(ep, df['lr/pg0'], color=COLORS['lr'], lw=2.5)
    ax6.fill_between(ep, df['lr/pg0'], alpha=0.2, color=COLORS['lr'])
    ax6.set_title('Learning Rate Schedule', fontsize=11, pad=6)
    ax6.set_ylabel('LR', fontsize=9); ax6.set_xlabel('Epoch', fontsize=9)
    ax6.grid(True, alpha=0.3)
    if len(ep) > 1: ax6.set_xlim(ep[0], max(ep[-1], total_epochs))

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


# ── YOLO 모니터링 루프 ────────────────────────────────────────
print(f'YOLO exp6 모니터링 시작 (갱신 주기: {REFRESH_SEC}초)')
print(f'결과 파일: {RESULTS_CSV}')
print('중단: ■ Stop 버튼')

while True:
    clear_output(wait=True)

    if not RESULTS_CSV.exists():
        print(f'대기 중... {RESULTS_CSV} 파일이 아직 없습니다.')
        time.sleep(5)
        continue

    df = pd.read_csv(RESULTS_CSV)
    df.columns = df.columns.str.strip()
    if len(df) == 0:
        print('데이터 대기 중...'); time.sleep(5); continue

    plot_yolo_dashboard(df, total_epochs=TOTAL_EPOCHS)

    latest = df.iloc[-1]
    n = len(df)
    if (EXP_DIR / 'weights' / 'best.pt').exists() and n >= TOTAL_EPOCHS:
        print('YOLO 학습 완료! best.pt 저장됨.')
        break

    print(f'[{time.strftime("%H:%M:%S")}]  Epoch {n}/{TOTAL_EPOCHS}  '
          f'mAP50={latest["metrics/mAP50(B)"]:.4f}  '
          f'다음 갱신까지 {REFRESH_SEC}초...')
    time.sleep(REFRESH_SEC)

## 완료 후 — 모델 검증 결과 보기

## MobileNetV3 Gaze Estimator 실시간 모니터링

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import clear_output
from pathlib import Path

# ── 경로 설정 ──────────────────────────────────────────────────
GAZE_RUNS  = Path('C:/Users/SSAFY/Desktop/GAZE-CAPTURE/GazeCapture/MPIIGAZE/runs/gaze')
GAZE_LOG   = GAZE_RUNS / 'gaze_train.log'   # raw log
TOTAL_EPOCHS = 30
REFRESH_SEC  = 30

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
    'font.size':        11,
})

GAZE_LOG_FILE = Path('C:/Users/SSAFY/Desktop/GAZE-CAPTURE/GazeCapture/MPIIGAZE/runs/gaze_train.log')


def find_latest_gaze_csv():
    """runs/gaze/runN/results.csv 중 최신 파일 반환"""
    gaze_dir = Path('C:/Users/SSAFY/Desktop/GAZE-CAPTURE/GazeCapture/MPIIGAZE/runs/gaze')
    csvs = sorted(gaze_dir.glob('run*/results.csv'))
    return csvs[-1] if csvs else None


def plot_gaze_dashboard(df: pd.DataFrame, total_epochs: int = 30):
    ep     = df['epoch'].values
    n      = len(df)
    latest = df.iloc[-1]

    fig = plt.figure(figsize=(16, 8))
    fig.patch.set_facecolor('#0d1117')
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

    pct = n / total_epochs * 100
    fig.suptitle(
        f'MobileNetV3 Gaze Estimator  |  Epoch {n}/{total_epochs}  ({pct:.0f}%)  |  '
        f'Val MAE {latest["val_mae_deg"]:.3f} deg',
        fontsize=14, fontweight='bold', color='#f0f6fc', y=0.98
    )

    # Progress bar
    ax0 = fig.add_subplot(gs[0, :])
    ax0.barh(0, 100, color='#21262d', height=0.6)
    ax0.barh(0, pct, color='#58a6ff', height=0.6)
    ax0.set_xlim(0, 100); ax0.set_yticks([])
    ax0.set_xlabel('에포크 진행률 (%)')
    for spine in ax0.spines.values(): spine.set_visible(False)
    best_mae = df['val_mae_deg'].min()
    best_ep  = df.loc[df['val_mae_deg'].idxmin(), 'epoch']
    ax0.text(50, -0.75,
        f"현재 Val MAE: {latest['val_mae_deg']:.3f} deg   "
        f"최고 MAE: {best_mae:.3f} deg (epoch {int(best_ep)})   "
        f"Train Loss: {latest['train_loss']:.5f}   "
        f"LR: {latest['lr']:.6f}",
        ha='center', va='top', color='#c9d1d9', fontsize=10)

    # Train/Val Loss
    ax1 = fig.add_subplot(gs[1, 0])
    ax1.plot(ep, df['train_loss'], color='#58a6ff', lw=2, label='Train')
    ax1.plot(ep, df['val_loss'],   color='#f78166', lw=2, label='Val', linestyle='--')
    ax1.fill_between(ep, df['train_loss'], alpha=0.15, color='#58a6ff')
    ax1.fill_between(ep, df['val_loss'],   alpha=0.15, color='#f78166')
    ax1.set_title('SmoothL1 Loss', fontsize=11, pad=6)
    ax1.set_ylabel('Loss', fontsize=9); ax1.set_xlabel('Epoch', fontsize=9)
    ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)
    if len(ep) > 1: ax1.set_xlim(ep[0], max(ep[-1], total_epochs))

    # Val MAE
    ax2 = fig.add_subplot(gs[1, 1])
    ax2.plot(ep, df['val_mae_deg'], color='#3fb950', lw=2.5)
    ax2.fill_between(ep, df['val_mae_deg'], alpha=0.2, color='#3fb950')
    ax2.axhline(best_mae, color='#ffa657', lw=1.5, linestyle=':', label=f'Best {best_mae:.3f}')
    ax2.set_title('Val MAE (각도 오차)', fontsize=11, pad=6)
    ax2.set_ylabel('MAE (degrees)', fontsize=9); ax2.set_xlabel('Epoch', fontsize=9)
    ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)
    if len(ep) > 1: ax2.set_xlim(ep[0], max(ep[-1], total_epochs))

    # LR
    ax3 = fig.add_subplot(gs[1, 2])
    ax3.plot(ep, df['lr'], color='#ffa657', lw=2.5)
    ax3.fill_between(ep, df['lr'], alpha=0.2, color='#ffa657')
    ax3.set_title('Learning Rate (Cosine)', fontsize=11, pad=6)
    ax3.set_ylabel('LR', fontsize=9); ax3.set_xlabel('Epoch', fontsize=9)
    ax3.grid(True, alpha=0.3)
    if len(ep) > 1: ax3.set_xlim(ep[0], max(ep[-1], total_epochs))

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


# ── Gaze 모니터링 루프 ─────────────────────────────────────────
print(f'MobileNetV3 Gaze 모니터링 시작 (갱신 주기: {REFRESH_SEC}초)')
print('중단: ■ Stop 버튼')

while True:
    clear_output(wait=True)
    csv_path = find_latest_gaze_csv()

    if csv_path is None or not csv_path.exists():
        # Show log status while training hasn't started
        if GAZE_LOG_FILE.exists():
            lines = GAZE_LOG_FILE.read_text(encoding='utf-8', errors='ignore').split('\n')
            non_bar = [l for l in lines if '|' not in l and l.strip()]
            print('\n'.join(non_bar[-8:]))
        else:
            print('Gaze 학습 대기 중 (runs/gaze/run*/results.csv 파일 없음)...')
        time.sleep(5)
        continue

    df = pd.read_csv(str(csv_path))
    df.columns = df.columns.str.strip()
    if len(df) == 0:
        print('데이터 대기 중...'); time.sleep(5); continue

    plot_gaze_dashboard(df, total_epochs=TOTAL_EPOCHS)

    latest = df.iloc[-1]
    n = len(df)

    # 완료 감지
    run_dir = csv_path.parent
    if (run_dir / 'best.pt').exists() and n >= TOTAL_EPOCHS:
        print(f'Gaze 학습 완료! 최고 MAE: {df["val_mae_deg"].min():.3f} deg')
        break

    print(f'[{time.strftime("%H:%M:%S")}]  Epoch {n}/{TOTAL_EPOCHS}  '
          f'Val MAE={latest["val_mae_deg"]:.3f} deg  '
          f'다음 갱신까지 {REFRESH_SEC}초...')
    time.sleep(REFRESH_SEC)

In [ ]:
# 학습 완료 후 최종 결과 이미지 표시
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

EXP_DIR = Path('C:/Users/SSAFY/Desktop/GAZE-CAPTURE/GazeCapture/MPIIGAZE/runs/yolo/exp4')

image_files = {
    'results.png':             '학습 곡선 (YOLO 자동 생성)',
    'confusion_matrix.png':    '혼동 행렬',
    'val_batch0_pred.jpg':     '검증 배치 예측 결과',
    'labels.jpg':              '라벨 분포',
}

for fname, title in image_files.items():
    fpath = EXP_DIR / fname
    if fpath.exists():
        img = mpimg.imread(str(fpath))
        fig, ax = plt.subplots(figsize=(14, 7))
        ax.imshow(img)
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f'아직 없음: {fname}')

In [ ]:
# 최종 성능 수치 요약
import pandas as pd
from pathlib import Path

csv = Path('C:/Users/SSAFY/Desktop/GAZE-CAPTURE/GazeCapture/MPIIGAZE/runs/yolo/exp4/results.csv')
df = pd.read_csv(csv)
df.columns = df.columns.str.strip()

best_idx = df['metrics/mAP50(B)'].idxmax()
best = df.loc[best_idx]

print('=' * 50)
print('  YOLOv8n 눈 탐지 — 최고 성능 에포크')
print('=' * 50)
print(f'  에포크        : {int(best["epoch"])}')
print(f'  mAP50         : {best["metrics/mAP50(B)"]:.4f}')
print(f'  mAP50-95      : {best["metrics/mAP50-95(B)"]:.4f}')
print(f'  Precision     : {best["metrics/precision(B)"]:.4f}')
print(f'  Recall        : {best["metrics/recall(B)"]:.4f}')
print(f'  Val Box Loss  : {best["val/box_loss"]:.4f}')
print(f'  Val Cls Loss  : {best["val/cls_loss"]:.4f}')
print('=' * 50)
print()
print('전체 에포크 기록:')
display(df[['epoch', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)',
            'metrics/precision(B)', 'metrics/recall(B)',
            'val/box_loss', 'val/cls_loss']].round(4))